# LeetCode 616 — Add Bold Tag in String bằng Python

Link đề: https://leetcode.com/problems/add-bold-tag-in-string/ · Độ khó: **Medium** · Chủ đề: Trie, Array, Hash Table, String, String Matching, Aho-Corasick

Skeleton LeetCode:

```python
class Solution:
    def addBoldTag(self, s: str, words: List[str]) -> str:
        ...
```

Ghi chú: đây là bài **Premium** (🔒). Đề trong notebook được viết theo hiểu biết về bài (và khớp với bài công khai tương đương 758. Bold Words in String), nên hãy đối chiếu lại đề gốc nếu bạn có tài khoản Premium. Ví dụ 3, 4, 5 trong mục 2 do notebook tự đặt, đáp án đã kiểm chứng bằng code; nếu cần khớp từng chữ, hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm mọi chỗ trong `s` mà một từ của `words` xuất hiện (kể cả **chồng lấn**), rồi bọc các đoạn đó bằng `<b>...</b>`; các đoạn chồng nhau **hoặc nằm sát nhau** phải gộp thành **một** cặp thẻ.
2. Nắm các công cụ Python: slicing `s[i:j]`, `str.find(sub, start)` và `str.startswith(sub, i)`, mảng đánh dấu (mask) kiểu `[False] * n`, `set`, `dict` làm trie, `''.join(list)`, `sorted`/hợp nhất khoảng (merge intervals).
3. Đi từ **thử mọi chuỗi con của `s`** → **đánh dấu từng ký tự (mask)** → **trie** → **quét một lượt với "tầm với xa nhất"** → **Aho-Corasick** (tìm nhiều mẫu trong một lượt).
4. Hiểu mấu chốt: bài thực chất hỏi "**ký tự nào của `s` nằm trong ít nhất một lần xuất hiện của một từ?**" — chỉ cần một mảng đúng/sai cho mỗi ký tự; thẻ chỉ cần chèn ở ranh giới giữa nhóm `True` liên tiếp và `False`. Hai điều kiện "chồng lấn" và "sát nhau" tự động được giải quyết.
5. Tránh các lỗi kinh điển: chỉ gộp khi chồng lấn mà quên gộp khi **sát nhau**, bỏ sót các lần xuất hiện chồng lấn (ví dụ `find` nhảy qua cả từ), dùng `replace` làm hỏng thẻ đã chèn, quên trường hợp `words` rỗng, và chèn thẻ lồng nhau.

## 2. Đề bài & ví dụ

Cho chuỗi `s` và mảng chuỗi `words`. Hãy thêm một cặp thẻ đậm `<b>` và `</b>` để bọc các **chuỗi con của `s` có mặt trong `words`**.

- Nếu hai chuỗi con như vậy **chồng lấn** (overlap), bọc chúng cùng nhau trong **một** cặp thẻ.
- Nếu hai chuỗi con đã được bọc **nằm sát nhau** (liên tiếp, không có ký tự nào ở giữa), hãy gộp chúng lại.

Trả về `s` sau khi thêm thẻ.

**Ví dụ 1**

- Input: `s = "abcxyz123"`, `words = ["abc","123"]`
- Output: `"<b>abc</b>xyz<b>123</b>"`

**Ví dụ 2**

- Input: `s = "aaabbb"`, `words = ["aa","b"]`
- Output: `"<b>aaabbb</b>"`
- Giải thích: `aa` xuất hiện tại `[0,1]` và `[1,2]` (chồng nhau → `aaa`), còn `b` xuất hiện tại vị trí 3, 4, 5. Đoạn `aaa` (0..2) và `b` đầu tiên (3..3) **sát nhau** nên gộp, cuối cùng cả chuỗi `aaabbb` nằm trong một cặp thẻ.

**Ví dụ 3 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `s = "abc"`, `words = []`
- Output: `"abc"` (không có từ nào nên không có thẻ).

**Ví dụ 4 (tự đặt)**

- Input: `s = "aaaa"`, `words = ["aaa"]`
- Output: `"<b>aaaa</b>"` (hai lần xuất hiện chồng lấn `[0,2]` và `[1,3]`).

**Ví dụ 5 (tự đặt)**

- Input: `s = "abcd"`, `words = ["ab","cd"]`
- Output: `"<b>abcd</b>"` (hai đoạn sát nhau nên gộp).

**Ràng buộc:**

- `1 <= s.length <= 1000`
- `0 <= words.length <= 100`
- `1 <= words[i].length <= 1000`
- `s` và `words[i]` chỉ gồm chữ cái tiếng Anh (hoa/thường) và chữ số.
- Các phần tử của `words` **đôi một khác nhau**.

Lưu ý: `words` có thể rỗng; phân biệt chữ hoa/thường.

In [1]:
from typing import List
from collections import deque
import random
import time

examples = [
    ("abcxyz123", ["abc", "123"], "<b>abc</b>xyz<b>123</b>"),
    ("aaabbb", ["aa", "b"], "<b>aaabbb</b>"),
    ("abc", [], "abc"),
    ("aaaa", ["aaa"], "<b>aaaa</b>"),
    ("abcd", ["ab", "cd"], "<b>abcd</b>"),
]
for s, words, expected in examples:
    print(f"s={s!r}, words={words} -> kỳ vọng {expected!r}")

s='abcxyz123', words=['abc', '123'] -> kỳ vọng '<b>abc</b>xyz<b>123</b>'
s='aaabbb', words=['aa', 'b'] -> kỳ vọng '<b>aaabbb</b>'
s='abc', words=[] -> kỳ vọng 'abc'
s='aaaa', words=['aaa'] -> kỳ vọng '<b>aaaa</b>'
s='abcd', words=['ab', 'cd'] -> kỳ vọng '<b>abcd</b>'


## 3. Kiến thức nền

### 3.1. Chuỗi con (substring) và slicing

Chuỗi con là một đoạn **liền nhau** của `s`. Với chỉ số `i <= j`, `s[i:j+1]` là chuỗi con bắt đầu ở `i` và kết thúc ở `j` (cả hai đều tính); nó có `j - i + 1` ký tự. Một chuỗi dài `n` có `n(n+1)/2` chuỗi con (tính theo vị trí).

In [2]:
s = "abcxyz123"
print(s[0:3], s[6:9], s[2:2 + 1])
n = len(s)
print("số chuỗi con (theo vị trí):", n * (n + 1) // 2, "=", sum(1 for i in range(n) for j in range(i, n)))

abc 123 c
số chuỗi con (theo vị trí): 45 = 45


### 3.2. Tìm mọi lần xuất hiện: `find` và `startswith`

- `s.find(w, start)` trả về chỉ số đầu tiên `>= start` mà `w` xuất hiện trong `s`, hoặc `-1`.
- `s.startswith(w, i)` hỏi "`w` có xuất hiện **ngay tại vị trí `i`** không".

Để tìm **cả những lần xuất hiện chồng nhau** (ví dụ `aa` trong `aaa` ở `0` và `1`), sau mỗi lần tìm thấy ở `p` phải tìm tiếp từ `p + 1`, **không** phải `p + len(w)`.

In [3]:
s, w = "aaa", "aa"
pos, p = [], s.find(w)
while p != -1:
    pos.append(p)
    p = s.find(w, p + 1)          # +1 chứ không phải +len(w)
print("tất cả lần xuất hiện:", pos)
print("bỏ sót nếu nhảy len(w):", [q for q in range(0, len(s), len(w)) if s.startswith(w, q)])
print([i for i in range(len(s)) if s.startswith(w, i)])

tất cả lần xuất hiện: [0, 1]
bỏ sót nếu nhảy len(w): [0]
[0, 1]


### 3.3. Mảng đánh dấu (mask) và gán theo lát cắt

Một list đúng/sai cho từng ký tự, `mask = [False] * n`, cho phép nhớ "ký tự này đã được bọc chưa". Python cho phép gán cả đoạn: `mask[a:b] = [True] * (b - a)`.

In [4]:
mask = [False] * 8
mask[2:5] = [True] * 3
mask[4:7] = [True] * 3          # chồng lên [4]
print(mask)
print("".join("#" if m else "." for m in mask))

[False, False, True, True, True, True, True, False]
..#####.


### 3.4. Gộp các đoạn thành "nhóm liên tiếp" và ghép chuỗi

Từ `mask`, một **khối** là dãy `True` liên tiếp tối đa. Ta chèn `<b>` trước khối và `</b>` sau khối. Khi ghép nhiều mẩu, thay vì `+=` chuỗi (chậm khi lặp lại) ta gom vào list rồi `''.join(list)`.

In [5]:
def render(s, mask, open_tag="<b>", close_tag="</b>"):
    out, i, n = [], 0, len(s)
    while i < n:
        if mask[i]:
            j = i
            while j < n and mask[j]:      # mở rộng khối True
                j += 1
            out.append(open_tag + s[i:j] + close_tag)
            i = j
        else:
            out.append(s[i])
            i += 1
    return "".join(out)

print(render("abcdefgh", [False, False, True, True, True, False, True, False]))
print(render("abcd", [True] * 4))
print(render("abcd", [False] * 4))

ab<b>cde</b>f<b>g</b>h
<b>abcd</b>
abcd


### 3.5. Hợp nhất khoảng (merge intervals)

Cho nhiều đoạn `[a, b]` (cả hai đầu tính), sắp theo `a`. Duyệt từ trái sang phải giữ đoạn hiện tại `[st, ed]`: đoạn mới `[a, b]` **gộp** vào nếu `a <= ed + 1` (chồng lấn **hoặc sát nhau**: `a = ed + 1`), lúc đó `ed = max(ed, b)`; ngược lại đóng đoạn hiện tại và bắt đầu đoạn mới. Điều kiện `a <= ed + 1` (chứ không phải `a <= ed`) là điểm khác bài merge intervals thông thường.

In [6]:
def merge_intervals(pairs):
    """pairs: list [a, b] (cả hai đầu tính), đã sắp theo a. Gộp khi chồng lấn hoặc sát nhau."""
    merged = []
    for a, b in pairs:
        if merged and a <= merged[-1][1] + 1:
            merged[-1][1] = max(merged[-1][1], b)
        else:
            merged.append([a, b])
    return merged

print(merge_intervals([[0, 1], [1, 2], [3, 3], [5, 6]]))    # [0,1],[1,2] chồng; [3,3] sát [0,2]; [5,6] tách rời
print(merge_intervals([[0, 2], [3, 5]]))
print(merge_intervals([[0, 2], [4, 5]]))

[[0, 3], [5, 6]]
[[0, 5]]
[[0, 2], [4, 5]]


### 3.6. `set` và `dict` làm trie

`set` kiểm tra thành viên `x in set` trung bình `O(1)`. **Trie** (cây tiền tố) là cây mà mỗi nút có các con theo ký tự; ta có thể dựng bằng `dict` lồng nhau: mỗi nút là một `dict {ký tự: nút con}` và dùng khoá đặc biệt (ở đây `"$"`) đánh dấu "kết thúc một từ". Duyệt từ vị trí `i` của `s` xuống trie cho ta **mọi từ bắt đầu ở `i`** trong một lượt.

In [7]:
def build_trie(words):
    root = {}
    for w in words:
        node = root
        for c in w:
            node = node.setdefault(c, {})
        node["$"] = True
    return root

trie = build_trie(["ab", "abc", "b"])
print(trie)
node = trie
for c in "abc":
    node = node[c]
    print(c, "kết thúc một từ?", "$" in node)

{'a': {'b': {'$': True, 'c': {'$': True}}}, 'b': {'$': True}}
a kết thúc một từ? False
b kết thúc một từ? True
c kết thúc một từ? True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy `s = "aaabbb"`, `words = ["aa", "b"]`. Con người làm thế này: **cầm bút dạ tô đậm** các ký tự của `s` mỗi khi một từ trùng với một đoạn.

| Vị trí | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| ký tự | a | a | a | b | b | b |
| `aa` tại 0..1 | ✔ | ✔ | | | | |
| `aa` tại 1..2 | | ✔ | ✔ | | | |
| `b` tại 3 | | | | ✔ | | |
| `b` tại 4 | | | | | ✔ | |
| `b` tại 5 | | | | | | ✔ |
| **đậm?** | ✔ | ✔ | ✔ | ✔ | ✔ | ✔ |

Cả sáu ký tự đều được tô nên ta chỉ cần **một** cặp thẻ bọc toàn bộ. Quan sát then chốt: ta **không cần biết ký tự nào thuộc từ nào**, chỉ cần biết ký tự **có được tô hay không**. Còn chuyện "chồng lấn" hay "sát nhau" chỉ là chuyện hai khối tô liền nhau — khi vẽ bằng bút dạ ta thấy chúng tự dính thành một khối, không phải làm gì thêm.

In [8]:
s, words = "aaabbb", ["aa", "b"]
bold = [False] * len(s)
for w in words:
    for i in range(len(s)):
        if s.startswith(w, i):
            for k in range(i, i + len(w)):
                bold[k] = True
print("".join(s))
print("".join("#" if b else "." for b in bold))
print(render(s, bold))

aaabbb
######
<b>aaabbb</b>


## 5. Cách 1 — Brute Force / baseline: thử mọi chuỗi con của `s`, rồi gộp khoảng

### 5.1. Ý tưởng

Cách trực tiếp nhất từ đề: với **mọi** cặp `(i, j)`, cắt `s[i:j+1]`, nếu nó có trong `set(words)` thì ghi lại khoảng `[i, j]`. Các khoảng sinh ra theo thứ tự tăng dần của `i` (rồi của `j`) nên đã **sắp xếp sẵn**; ta dùng `merge_intervals` ở 3.5 (gộp cả sát nhau), rồi cắt-ghép chuỗi kết quả.

In [9]:
def add_bold_brute(s, words):
    wset = set(words)
    n = len(s)
    pairs = []
    for i in range(n):
        for j in range(i, n):
            if s[i:j + 1] in wset:
                pairs.append([i, j])
    merged = merge_intervals(pairs)
    out, prev = [], 0
    for a, b in merged:
        out.append(s[prev:a])
        out.append("<b>" + s[a:b + 1] + "</b>")
        prev = b + 1
    out.append(s[prev:])
    return "".join(out)

for s, words, expected in examples:
    got = add_bold_brute(s, words)
    print(f"{s!r}, {words} -> {got!r}")
    assert got == expected

'abcxyz123', ['abc', '123'] -> '<b>abc</b>xyz<b>123</b>'
'aaabbb', ['aa', 'b'] -> '<b>aaabbb</b>'
'abc', [] -> 'abc'
'aaaa', ['aaa'] -> '<b>aaaa</b>'
'abcd', ['ab', 'cd'] -> '<b>abcd</b>'


### 5.2. Độ phức tạp và nút thắt

- Số cặp `(i, j)` là `O(n²)`, mỗi lần cắt và băm chuỗi tốn `O(độ dài)` → Time `O(n³)` (tệ nhất với `n = 1000` là cỡ `10^9` thao tác ký tự), Space `O(n²)` cho danh sách khoảng trong trường hợp xấu.

Nút thắt: ta thử **cả những chuỗi con dài hơn mọi từ** trong `words`, và dựng hàng nghìn chuỗi con mới chỉ để tra `set`. Hãy đo:

In [10]:
rng = random.Random(616)
for n in (100, 200, 400):
    s = "".join(rng.choice("ab") for _ in range(n))
    t0 = time.perf_counter()
    add_bold_brute(s, ["abab", "bba"])
    print(f"n={n:3d}: {1000 * (time.perf_counter() - t0):8.2f} ms")

n=100:     0.51 ms
n=200:     2.09 ms
n=400:    11.94 ms


## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 — đổi câu hỏi.** Thay vì quản lý các khoảng rồi gộp, hãy hỏi: *"ký tự `k` có bị bọc không?"* Ký tự `k` bị bọc ⇔ tồn tại lần xuất hiện `[i, j]` của một từ với `i <= k <= j`. Lưu kết quả vào mảng `bold[0..n-1]`.

**Bước 2 — đọc đáp án từ mảng.** Chỉ cần chèn `<b>` ở chỗ `bold` đổi từ `False` sang `True` và `</b>` ở chỗ đổi từ `True` sang `False` (hoặc hết chuỗi). Nhờ vậy **chồng lấn** (hai khoảng giao nhau) và **sát nhau** (hai khoảng liền kề) đều cho ra các ô `True` liên tiếp và tự gộp — ta không cần điều kiện `ed + 1` nào cả.

**Bước 3 — tăng tốc việc tìm các lần xuất hiện.** Tìm mọi lần xuất hiện của mọi từ có thể làm theo nhiều cách: `startswith` tại từng vị trí, `find` lặp lại, **trie** quét một lượt cho mỗi vị trí bắt đầu, hoặc **Aho-Corasick** quét `s` **đúng một lần** cho mọi từ cùng lúc.

**Bước 4 — một mẹo nhỏ về mask.** Với mỗi vị trí bắt đầu `i`, chỉ cần biết **lần xuất hiện dài nhất bắt đầu ở `i`** (các lần ngắn hơn bắt đầu ở `i` bị nó bao trùm). Nếu đặt `reach[i] = i + (độ dài từ dài nhất bắt đầu ở i)` thì `bold[k]` đúng ⇔ `max(reach[0..k]) > k`. Đây là cách 4.

## 7. Các cách tiếp theo

Tổng cộng có 5 cách: Cách 1 (thử mọi chuỗi con, mục 5) và bốn cách dưới đây. Mỗi cách: giải thích từng đoạn → code demo → code nộp LeetCode → độ phức tạp.

### 7.1. Cách 2 — Đánh dấu mask theo từng từ

Giải thích từng đoạn:

1. `bold = [False] * n`.
2. Với mỗi từ `w`: dùng `s.find(w, start)` tìm các lần xuất hiện; mỗi lần xuất hiện tại `p` thì `bold[p : p + len(w)] = [True] * len(w)`; rồi tìm tiếp từ `p + 1` (cho phép chồng lấn).
3. `render` mask thành chuỗi kết quả (mục 3.4).

In [11]:
def mask_demo(s, words):
    bold = [False] * len(s)
    for w in words:
        p = s.find(w)
        while p != -1:
            bold[p:p + len(w)] = [True] * len(w)
            print(f"  từ {w!r} tại {p}: bold = {''.join('#' if b else '.' for b in bold)}")
            p = s.find(w, p + 1)
    return render(s, bold)

print(mask_demo("aaabbb", ["aa", "b"]))

  từ 'aa' tại 0: bold = ##....
  từ 'aa' tại 1: bold = ###...
  từ 'b' tại 3: bold = ####..
  từ 'b' tại 4: bold = #####.
  từ 'b' tại 5: bold = ######
<b>aaabbb</b>


In [12]:
class Sol2:
    """Cách 2: đánh dấu mask bằng find."""
    def addBoldTag(self, s: str, words: List[str]) -> str:
        n = len(s)
        bold = [False] * n
        for w in words:
            p = s.find(w)
            while p != -1:
                bold[p:p + len(w)] = [True] * len(w)
                p = s.find(w, p + 1)
        out, i = [], 0
        while i < n:
            if bold[i]:
                j = i
                while j < n and bold[j]:
                    j += 1
                out.append("<b>" + s[i:j] + "</b>")
                i = j
            else:
                out.append(s[i])
                i += 1
        return "".join(out)

for s, words, expected in examples:
    assert Sol2().addBoldTag(s, words) == expected
print("Sol2 OK")

Sol2 OK


**Độ phức tạp:** gọi `W` là số từ, `L` độ dài từ tối đa. `find` mỗi lần tối đa `O(n · L)` (so khớp thô) và có thể lặp tới `n` lần cho mỗi từ; gán lát cắt tốn tổng cộng `O(n · L)` mỗi từ. Tệ nhất `O(W · n · L)`; với `W ≤ 100`, `n ≤ 1000` và các từ thực tế không dài, chạy tốt. Space `O(n)`.

### 7.2. Cách 3 — Trie: quét từng vị trí bắt đầu (cách trong lời giải mẫu)

Giải thích từng đoạn:

1. Dựng trie từ `words` (mục 3.6).
2. Với mỗi vị trí bắt đầu `i`, đi xuống trie theo `s[i], s[i+1], ...`; **dừng ngay** khi không còn nhánh. Mỗi lần gặp nút kết thúc từ tại `j` ta ghi khoảng `[i, j]`.
3. Danh sách khoảng đã sắp theo `i`; `merge_intervals` (gộp cả sát nhau), rồi cắt-ghép chuỗi.

Ưu điểm: không dò lại từng từ riêng lẻ; chi phí mỗi vị trí chỉ bằng độ dài từ khớp dài nhất bắt đầu ở đó (không bằng độ dài toàn bộ chuỗi con).

In [13]:
def trie_demo(s, words):
    trie = build_trie(words)
    pairs = []
    for i in range(len(s)):
        node = trie
        trail = []
        for j in range(i, len(s)):
            if s[j] not in node:
                break
            node = node[s[j]]
            trail.append(s[j])
            if "$" in node:
                pairs.append([i, j])
        print(f"  i={i}: đi được {''.join(trail)!r}")
    print("  khoảng:", pairs, "-> gộp:", merge_intervals(pairs))

trie_demo("aaabbb", ["aa", "b"])

  i=0: đi được 'aa'
  i=1: đi được 'aa'
  i=2: đi được 'a'
  i=3: đi được 'b'
  i=4: đi được 'b'
  i=5: đi được 'b'
  khoảng: [[0, 1], [1, 2], [3, 3], [4, 4], [5, 5]] -> gộp: [[0, 5]]


In [14]:
class Sol3:
    """Cách 3: trie + gộp khoảng."""
    def addBoldTag(self, s: str, words: List[str]) -> str:
        root = {}
        for w in words:
            node = root
            for c in w:
                node = node.setdefault(c, {})
            node["$"] = True
        n = len(s)
        pairs = []
        for i in range(n):
            node = root
            for j in range(i, n):
                if s[j] not in node:
                    break
                node = node[s[j]]
                if "$" in node:
                    pairs.append([i, j])
        merged = []
        for a, b in pairs:
            if merged and a <= merged[-1][1] + 1:
                merged[-1][1] = max(merged[-1][1], b)
            else:
                merged.append([a, b])
        out, prev = [], 0
        for a, b in merged:
            out.append(s[prev:a])
            out.append("<b>" + s[a:b + 1] + "</b>")
            prev = b + 1
        out.append(s[prev:])
        return "".join(out)

for s, words, expected in examples:
    assert Sol3().addBoldTag(s, words) == expected
print("Sol3 OK")

Sol3 OK


**Độ phức tạp:** dựng trie `O(Σ|w|)`; quét `O(n · L)` (mỗi vị trí đi tối đa `L` bước, `L` là độ dài từ dài nhất); gộp `O(số khoảng)`. Space `O(Σ|w|)` cho trie cộng danh sách khoảng. Lưu ý: `"$"` an toàn làm khoá đánh dấu vì `s` và `words` chỉ gồm chữ cái/chữ số.

### 7.3. Cách 4 — Quét một lượt với "tầm với xa nhất" (running max, không cần mask riêng)

Ý tưởng ở mục 6 bước 4: gọi `reach[i]` là `i + (độ dài từ dài nhất khớp tại i)` (hoặc `0` nếu không có từ nào khớp ở `i`). Duyệt `k` từ trái sang phải, giữ `far = max(reach[0..k])`; ký tự `k` được tô ⇔ `k < far`.

Vì sao đúng: `k` bị bao phủ bởi lần xuất hiện `[i, j]` ⇔ `i <= k <= j`, mà tồn tại `i <= k` với `reach[i] > k` ⇔ `max_{i<=k} reach[i] > k`. Chỉ lấy lần khớp **dài nhất** tại mỗi `i` là đủ vì các lần ngắn hơn cùng điểm bắt đầu nằm trọn trong nó.

In [15]:
def reach_demo(s, words):
    n = len(s)
    reach = [0] * n
    for i in range(n):
        for w in words:
            if s.startswith(w, i):
                reach[i] = max(reach[i], i + len(w))
    print("  reach =", reach)
    far, bold = 0, []
    for k in range(n):
        far = max(far, reach[k])
        bold.append(k < far)
        print(f"  k={k} ({s[k]}): far={far} -> {'tô' if k < far else 'không'}")
    return render(s, bold)

print(reach_demo("aaabbb", ["aa", "b"]))

  reach = [2, 3, 0, 4, 5, 6]
  k=0 (a): far=2 -> tô
  k=1 (a): far=3 -> tô
  k=2 (a): far=3 -> tô
  k=3 (b): far=4 -> tô
  k=4 (b): far=5 -> tô
  k=5 (b): far=6 -> tô
<b>aaabbb</b>


In [16]:
class Sol4:
    """Cách 4: reach[i] + running max."""
    def addBoldTag(self, s: str, words: List[str]) -> str:
        n = len(s)
        reach = [0] * n
        for i in range(n):
            for w in words:
                if s.startswith(w, i):
                    reach[i] = max(reach[i], i + len(w))
        out, far = [], 0
        in_bold = False
        for k in range(n):
            far = max(far, reach[k])
            if k < far and not in_bold:
                out.append("<b>")
                in_bold = True
            elif k >= far and in_bold:
                out.append("</b>")
                in_bold = False
            out.append(s[k])
        if in_bold:
            out.append("</b>")
        return "".join(out)

for s, words, expected in examples:
    assert Sol4().addBoldTag(s, words) == expected
print("Sol4 OK")

Sol4 OK


**Độ phức tạp:** Time `O(n · W · L)` cho `startswith` (có thể thay bằng trie để còn `O(n · L)`), phần quét `O(n)`; Space `O(n)`. Điểm hay: chèn thẻ **ngay lúc quét**, không cần mảng `bold` hay `merge`.

### 7.4. Cách 5 — Aho-Corasick: quét `s` đúng một lần cho mọi từ

Aho-Corasick là trie bổ sung **liên kết thất bại** (failure link): khi đang ở nút ứng với chuỗi `u` mà ký tự tiếp theo không có nhánh, nhảy về nút ứng với **hậu tố dài nhất của `u` còn là tiền tố của một từ**. Nhờ vậy ta đọc `s` từ trái sang phải **không lùi**, và tại mỗi vị trí `j` biết ngay mọi từ **kết thúc** ở `j`.

Các bước:

1. Dựng trie `go` (mỗi nút là một dict), ghi `own[v] = len(w)` ở nút kết thúc từ.
2. BFS theo tầng để tính `fail[v]` và `best[v] = max(own[v], best[fail[v]])` — độ dài từ dài nhất **kết thúc** khi ở nút `v` (kể cả từ là hậu tố, đi qua chuỗi liên kết thất bại).
3. Quét `s`: chuyển trạng thái, rồi nếu `best[state] = L > 0` thì đoạn `[j - L + 1, j]` bị bao phủ. Dùng **mảng hiệu** `diff[j-L+1] += 1`, `diff[j+1] -= 1` rồi cộng dồn để biết ký tự nào có độ phủ `> 0`. (Chỉ lấy lần dài nhất kết thúc ở `j` là đủ vì các lần ngắn hơn cùng điểm cuối nằm trọn trong nó.)

In [17]:
def build_ac(words):
    go, fail, own = [{}], [0], [0]
    for w in words:
        v = 0
        for c in w:
            if c not in go[v]:
                go.append({})
                fail.append(0)
                own.append(0)
                go[v][c] = len(go) - 1
            v = go[v][c]
        own[v] = len(w)
    best = own[:]
    queue = deque(go[0].values())                 # nút độ sâu 1: fail = 0
    while queue:
        v = queue.popleft()
        for c, u in go[v].items():
            f = fail[v]
            while f and c not in go[f]:
                f = fail[f]
            fail[u] = go[f].get(c, 0)
            best[u] = max(best[u], best[fail[u]])
            queue.append(u)
    return go, fail, best

go, fail, best = build_ac(["aa", "b", "ab"])
print("go  :", go)
print("fail:", fail)
print("best:", best)

go  : [{'a': 1, 'b': 3}, {'a': 2, 'b': 4}, {}, {}, {}]
fail: [0, 0, 1, 0, 3]
best: [0, 0, 2, 1, 2]


In [18]:
def ac_demo(s, words):
    go, fail, best = build_ac(words)
    n, state = len(s), 0
    diff = [0] * (n + 1)
    for j, c in enumerate(s):
        while state and c not in go[state]:
            state = fail[state]
        state = go[state].get(c, 0)
        L = best[state]
        print(f"  j={j} ({c}): state={state} best={L}" + (f" -> phủ [{j - L + 1},{j}]" if L else ""))
        if L:
            diff[j - L + 1] += 1
            diff[j + 1] -= 1
    cur, bold = 0, []
    for k in range(n):
        cur += diff[k]
        bold.append(cur > 0)
    return render(s, bold)

print(ac_demo("aaabbb", ["aa", "b"]))

  j=0 (a): state=1 best=0
  j=1 (a): state=2 best=2 -> phủ [0,1]
  j=2 (a): state=2 best=2 -> phủ [1,2]
  j=3 (b): state=3 best=1 -> phủ [3,3]
  j=4 (b): state=3 best=1 -> phủ [4,4]
  j=5 (b): state=3 best=1 -> phủ [5,5]
<b>aaabbb</b>


In [19]:
class Sol5:
    """Cách 5: Aho-Corasick + mảng hiệu."""
    def addBoldTag(self, s: str, words: List[str]) -> str:
        from collections import deque
        go, fail, best = [{}], [0], [0]
        for w in words:
            v = 0
            for c in w:
                if c not in go[v]:
                    go.append({})
                    fail.append(0)
                    best.append(0)
                    go[v][c] = len(go) - 1
                v = go[v][c]
            best[v] = len(w)
        queue = deque(go[0].values())
        while queue:
            v = queue.popleft()
            for c, u in go[v].items():
                f = fail[v]
                while f and c not in go[f]:
                    f = fail[f]
                fail[u] = go[f].get(c, 0)
                best[u] = max(best[u], best[fail[u]])
                queue.append(u)

        n = len(s)
        diff = [0] * (n + 1)
        state = 0
        for j, c in enumerate(s):
            while state and c not in go[state]:
                state = fail[state]
            state = go[state].get(c, 0)
            L = best[state]
            if L:
                diff[j - L + 1] += 1
                diff[j + 1] -= 1
        out, cur, in_bold = [], 0, False
        for k in range(n):
            cur += diff[k]
            if cur > 0 and not in_bold:
                out.append("<b>")
                in_bold = True
            elif cur == 0 and in_bold:
                out.append("</b>")
                in_bold = False
            out.append(s[k])
        if in_bold:
            out.append("</b>")
        return "".join(out)

for s, words, expected in examples:
    assert Sol5().addBoldTag(s, words) == expected
print("Sol5 OK")

Sol5 OK


**Độ phức tạp:** dựng `O(Σ|w|)` (số lần nhảy `fail` được khấu hao), quét `s` `O(n)` khấu hao, mảng hiệu `O(n)`. Tổng `O(n + Σ|w|)` — nhanh nhất về tiệm cận, hợp khi có **rất nhiều từ** hoặc `s` rất dài. Space `O(Σ|w| + n)`. Với giới hạn đề nhỏ, Aho-Corasick là "dùng dao mổ trâu", nhưng là công cụ chuẩn của bài toán "tìm nhiều mẫu cùng lúc".

## 8. TRACE từng bước

### 8.1. Trace Cách 1: gộp khoảng (bao gồm trường hợp sát nhau)

Với `s = "aaabbb"`, `words = ["aa", "b"]` ta tìm các khoảng `[i, j]`: `[0,1]`, `[1,2]`, `[3,3]`, `[4,4]`, `[5,5]`. Bảng gộp (đoạn hiện tại `[st, ed]`):

| đoạn mới `[a, b]` | điều kiện `a <= ed + 1` | hành động | danh sách gộp |
|---|---|---|---|
| `[0,1]` | (đầu) | mở `[0,1]` | `[[0,1]]` |
| `[1,2]` | `1 <= 2` đúng | `ed = max(1,2) = 2` | `[[0,2]]` |
| `[3,3]` | `3 <= 3` đúng (**sát nhau**) | `ed = 3` | `[[0,3]]` |
| `[4,4]` | `4 <= 4` đúng | `ed = 4` | `[[0,4]]` |
| `[5,5]` | `5 <= 5` đúng | `ed = 5` | `[[0,5]]` |

In [20]:
def trace_merge(s, words):
    wset = set(words)
    pairs = [[i, j] for i in range(len(s)) for j in range(i, len(s)) if s[i:j + 1] in wset]
    print("khoảng thô:", pairs)
    merged = []
    for a, b in pairs:
        if merged and a <= merged[-1][1] + 1:
            why = "chồng lấn" if a <= merged[-1][1] else "sát nhau"
            old = list(merged[-1])
            merged[-1][1] = max(merged[-1][1], b)
            print(f"  [{a},{b}] {why} với {old} -> {merged[-1]}")
        else:
            merged.append([a, b])
            print(f"  [{a},{b}] khoảng mới")
    return merged

trace_merge("aaabbb", ["aa", "b"])
trace_merge("abcxyz123", ["abc", "123"])

khoảng thô: [[0, 1], [1, 2], [3, 3], [4, 4], [5, 5]]
  [0,1] khoảng mới
  [1,2] chồng lấn với [0, 1] -> [0, 2]
  [3,3] sát nhau với [0, 2] -> [0, 3]
  [4,4] sát nhau với [0, 3] -> [0, 4]
  [5,5] sát nhau với [0, 4] -> [0, 5]
khoảng thô: [[0, 2], [6, 8]]
  [0,2] khoảng mới
  [6,8] khoảng mới


[[0, 2], [6, 8]]

Quan sát: hai khoảng chỉ gộp khi `a <= ed + 1`. Với `abcxyz123`, `[0,2]` và `[6,8]` cách nhau (`6 > 3`) nên giữ hai khối.

### 8.2. Trace Cách 2: mask sau từng lần xuất hiện

Ta chạy `mask_demo` cho ví dụ có chồng lấn nhẹ và sát nhau:

In [21]:
print(mask_demo("abcdabc", ["ab", "bc", "cd"]))
print()
print(mask_demo("xaaax", ["aa"]))

  từ 'ab' tại 0: bold = ##.....
  từ 'ab' tại 4: bold = ##..##.
  từ 'bc' tại 1: bold = ###.##.
  từ 'bc' tại 5: bold = ###.###
  từ 'cd' tại 2: bold = #######
<b>abcdabc</b>

  từ 'aa' tại 1: bold = .##..
  từ 'aa' tại 2: bold = .###.
x<b>aaa</b>x


### 8.3. Trace Cách 3: trie đi sâu bao xa

`trie_demo` đã in ở mục 7.2 cho `aaabbb`. Với một ví dụ nhiều từ chung tiền tố (`ab`, `abc`, `abcd`, `bc`), ta thấy mỗi vị trí `i` có thể ghi nhiều khoảng:

In [22]:
trie_demo("zabcdz", ["ab", "abc", "abcd", "bc"])

  i=0: đi được ''
  i=1: đi được 'abcd'
  i=2: đi được 'bc'
  i=3: đi được ''
  i=4: đi được ''
  i=5: đi được ''
  khoảng: [[1, 2], [1, 3], [1, 4], [2, 3]] -> gộp: [[1, 4]]


### 8.4. Trace Cách 4: bảng `reach` và `far`

Mỗi dòng của `reach_demo` cho thấy `far` (độ phủ tối đa tính từ các vị trí bắt đầu đến `k`) và ký tự `k` có được tô không. Ví dụ có khoảng trống:

In [23]:
print(reach_demo("abxxcd", ["ab", "cd"]))

  reach = [2, 0, 0, 0, 6, 0]
  k=0 (a): far=2 -> tô
  k=1 (b): far=2 -> tô
  k=2 (x): far=2 -> không
  k=3 (x): far=2 -> không
  k=4 (c): far=6 -> tô
  k=5 (d): far=6 -> tô
<b>ab</b>xx<b>cd</b>


Ở `k = 2, 3` (`xx`): `far = 2 <= k` nên **không** tô và thẻ `</b>` được chèn trước `x` đầu tiên; ở `k = 4` `reach[4] = 6` mở khối mới.

### 8.5. Trace Cách 5: các trạng thái Aho-Corasick

Với `words = ["aa", "ab"]`, `s = "aab"`: trạng thái đi theo trie `a → aa` ở ký tự thứ hai, sang ký tự `b` thì không có nhánh `aab`, nên nhảy theo `fail`.

In [24]:
print(ac_demo("aab", ["aa", "ab"]))
go, fail, best = build_ac(["aa", "ab"])
print("go:", go, "| fail:", fail, "| best:", best)

  j=0 (a): state=1 best=0
  j=1 (a): state=2 best=2 -> phủ [0,1]
  j=2 (b): state=3 best=2 -> phủ [1,2]
<b>aab</b>
go: [{'a': 1}, {'a': 2, 'b': 3}, {}, {}] | fail: [0, 0, 1, 0] | best: [0, 0, 2, 2]


Khi ở nút `aa` (trạng thái 2) và gặp `b`: nút này không có con `b`, nhảy `fail[2] = 1` (nút `a`), nút `a` có con `b` → tới nút `ab` (`best = 2`) → phủ `[1, 2]`; cùng với `[0, 1]` từ `aa` được gộp thành `<b>aab</b>`.

## 9. Code nộp LeetCode

Dán đoạn dưới vào LeetCode (bài Premium; trên LeetCode đã có sẵn `List`). Ta nộp cách 3 (trie + gộp khoảng) vì cân bằng giữa độ dài code và hiệu năng.

In [25]:
class Solution:
    def addBoldTag(self, s: str, words: List[str]) -> str:
        # 1) dựng trie
        root = {}
        for w in words:
            node = root
            for c in w:
                node = node.setdefault(c, {})
            node["$"] = True

        # 2) mọi khoảng [i, j] mà s[i..j] là một từ
        n = len(s)
        pairs = []
        for i in range(n):
            node = root
            for j in range(i, n):
                if s[j] not in node:
                    break
                node = node[s[j]]
                if "$" in node:
                    pairs.append([i, j])
        if not pairs:
            return s

        # 3) gộp khi chồng lấn hoặc sát nhau (a <= ed + 1)
        merged = []
        for a, b in pairs:
            if merged and a <= merged[-1][1] + 1:
                merged[-1][1] = max(merged[-1][1], b)
            else:
                merged.append([a, b])

        # 4) cắt - ghép chuỗi
        out, prev = [], 0
        for a, b in merged:
            out.append(s[prev:a])
            out.append("<b>" + s[a:b + 1] + "</b>")
            prev = b + 1
        out.append(s[prev:])
        return "".join(out)


for s, words, expected in examples:
    assert Solution().addBoldTag(s, words) == expected
print("Solution chạy đúng 5 ví dụ")

Solution chạy đúng 5 ví dụ


## 10. Vì sao thuật toán đúng

Gọi `occ` là tập mọi khoảng `[i, j]` mà `s[i..j]` thuộc `words`. Đáp án đúng theo đề là: lấy **hợp** các khoảng trong `occ` rồi mỗi **thành phần liên thông tối đa** (các ký tự tô liền nhau) được bọc bởi một cặp thẻ.

**Bổ đề 1 (hai điều kiện của đề = "liền nhau trên mask").** Hai khoảng bị bọc chung một cặp thẻ khi chồng lấn (`a <= ed`) hoặc sát nhau (`a = ed + 1`). Hai trường hợp đó đúng bằng "hợp của hai khoảng là một đoạn **liên tục** các vị trí". Do đó đáp án = các đoạn liên tục tối đa của hợp, chính là các khối `True` liên tiếp tối đa của mảng `bold` với `bold[k] = (∃ [i, j] ∈ occ : i ≤ k ≤ j)`.

**Cách 2, 4, 5** tính trực tiếp `bold` (hoặc tương đương) nên đúng theo Bổ đề 1; chỉ cần thấy rằng mỗi cách bao phủ đúng các `k` đó:

- Cách 2: gán `True` cho `[p, p + len(w) - 1]` với **mọi** `p` mà `w` xuất hiện (`find` lặp từ `p + 1` nên không sót lần chồng lấn).
- Cách 4: `k` được tô ⇔ `max_{i ≤ k} reach[i] > k` (đã chứng minh ở 7.3).
- Cách 5: tại `j`, `best[state]` là độ dài từ dài nhất kết thúc ở `j`. Điều này đúng vì chuỗi đã đọc `s[..j]` có hậu tố dài nhất là tiền tố của trie ở `state`; mọi từ kết thúc ở `j` là một hậu tố của chuỗi đó và là tiền tố của trie nên nằm trên chuỗi `fail` của `state`. Các khoảng ngắn hơn cùng điểm cuối nằm trọn trong khoảng dài nhất nên không cần ghi.

**Cách 1 và 3** dùng gộp khoảng: danh sách `pairs` được sinh theo `i` tăng dần (và `j` tăng dần với cùng `i`), tức đã sắp theo đầu trái. Với danh sách khoảng sắp theo đầu trái, thuật toán giữ khoảng hiện tại `[st, ed]` và gộp `[a, b]` khi `a <= ed + 1` cho ra đúng các thành phần liên tục tối đa của hợp (chuẩn của merge intervals, ở đây "liên tục" tính cả sát nhau): nếu `a > ed + 1` thì vị trí `ed + 1` không thuộc khoảng nào có đầu trái `<= ed` (do đã sắp) và cũng không thuộc khoảng nào đầu trái `>= a` (đều `> ed + 1`), nên ngăn cách hai khối. ∎

**Trie đảm bảo không bỏ sót:** từ vị trí `i`, đi theo trie gặp đúng mọi từ bắt đầu ở `i` (từ nào là tiền tố của `s[i:]` thì nằm trên đường đi); dừng khi không còn nhánh là an toàn vì không từ nào dài hơn còn khớp.

**Về thẻ:** ta chỉ chèn thẻ quanh các khối tối đa nên không bao giờ có thẻ lồng hoặc hai khối kề sát nhau — đầu ra luôn có dạng chuẩn.

## 11. Phân tích Time / Space Complexity

Ký hiệu: `n = |s|`, `W` = số từ, `L` = độ dài từ lớn nhất, `T = Σ|w|` (tổng độ dài các từ).

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi chuỗi con + gộp khoảng | `O(n² · L)` (băm chuỗi con) | `O(n²)` |
| 2. Mask theo từng từ (`find`) | `O(W · n · L)` | `O(n)` |
| 3. Trie + gộp khoảng | `O(T + n · L)` | `O(T + n)` |
| 4. `reach` + running max | `O(n · W · L)` (hoặc `O(T + n · L)` nếu dùng trie) | `O(n)` |
| 5. Aho-Corasick + mảng hiệu | `O(T + n)` khấu hao | `O(T + n)` |

Với giới hạn đề (`n ≤ 1000`, `W ≤ 100`), mọi cách (trừ Cách 1 ở mức xấu nhất) đều nhanh. Ta đo thực nghiệm trên một đầu vào lớn nhất:

In [26]:
rng = random.Random(6160)
S = "".join(rng.choice("abc") for _ in range(1000))
W = list({"".join(rng.choice("abc") for _ in range(rng.randint(1, 8))) for _ in range(100)})
W = [w for w in W][:100]
solvers_big = [("Cách 2", Sol2), ("Cách 3", Sol3), ("Cách 4", Sol4), ("Cách 5", Sol5), ("Solution", Solution)]
outs = {}
for name, cls in solvers_big:
    t0 = time.perf_counter()
    outs[name] = cls().addBoldTag(S, W)
    print(f"{name:<9}: {1000 * (time.perf_counter() - t0):8.2f} ms")
assert len(set(outs.values())) == 1
print("kết quả giống nhau; số thẻ <b>:", outs["Cách 2"].count("<b>"))

Cách 2   :     1.33 ms
Cách 3   :     1.66 ms
Cách 4   :     4.88 ms
Cách 5   :     0.78 ms
Solution :     1.23 ms
kết quả giống nhau; số thẻ <b>: 1


## 12. Bộ test

### 12.1. Ví dụ đề và edge case

**Oracle độc lập:** với **từng ký tự** `k`, kiểm tra bằng vét cạn xem tồn tại chuỗi con `s[i:j+1]` bao `k` mà thuộc `set(words)`; sau đó dựng kết quả bằng cách đi qua từng ký tự và chèn `<b>` khi **chuyển** từ không bold sang bold và `</b>` khi chuyển ngược lại. Không dùng `find`, trie, gộp khoảng, running max hay Aho-Corasick. Ta gom các cách vào dict `solvers`.

In [27]:
def oracle(s, words):
    wset = set(words)
    n = len(s)
    bold = [any(s[i:j + 1] in wset for i in range(k + 1) for j in range(k, n)) for k in range(n)]
    out = ""
    for k in range(n):
        if bold[k] and (k == 0 or not bold[k - 1]):
            out += "<b>"
        if not bold[k] and k > 0 and bold[k - 1]:
            out += "</b>"
        out += s[k]
    if n and bold[-1]:
        out += "</b>"
    return out

solvers = {
    "brute": add_bold_brute,
    "mask": lambda s, w: Sol2().addBoldTag(s, w),
    "trie": lambda s, w: Sol3().addBoldTag(s, w),
    "reach": lambda s, w: Sol4().addBoldTag(s, w),
    "aho": lambda s, w: Sol5().addBoldTag(s, w),
    "leetcode": lambda s, w: Solution().addBoldTag(s, w),
}

for s, words, expected in examples:
    assert oracle(s, words) == expected
    for name, fn in solvers.items():
        assert fn(s, words) == expected, (name, s, words)
print("Ví dụ đề: OK cho cả 6 solver và oracle")

Ví dụ đề: OK cho cả 6 solver và oracle


In [28]:
edge_cases = [
    ("a", [], "a", "words rỗng"),
    ("a", ["a"], "<b>a</b>", "một ký tự khớp"),
    ("a", ["b"], "a", "không khớp"),
    ("a", ["aa"], "a", "từ dài hơn s"),
    ("ab", ["ab"], "<b>ab</b>", "từ = s"),
    ("ab", ["a", "b"], "<b>ab</b>", "hai từ sát nhau"),
    ("abc", ["a", "c"], "<b>a</b>b<b>c</b>", "hai khối tách rời bởi 1 ký tự"),
    ("aaaa", ["a"], "<b>aaaa</b>", "mỗi ký tự là một lần xuất hiện"),
    ("aaaa", ["aa"], "<b>aaaa</b>", "chồng lấn"),
    ("aaaaa", ["aa", "aaa"], "<b>aaaaa</b>", "nhiều từ cùng tiền tố"),
    ("abcxyz123", ["abc", "123"], "<b>abc</b>xyz<b>123</b>", "ví dụ 1"),
    ("abcxyz123", ["abc", "xyz", "123"], "<b>abcxyz123</b>", "ba khối sát nhau"),
    ("Abc", ["abc"], "Abc", "phân biệt hoa/thường"),
    ("Abc", ["Abc", "bc"], "<b>Abc</b>", "từ bao từ"),
    ("a1b2", ["1", "2"], "a<b>1</b>b<b>2</b>", "chữ số"),
    ("abcd", ["bc"], "a<b>bc</b>d", "khối ở giữa"),
    ("abcd", ["abcd", "bc"], "<b>abcd</b>", "từ chứa từ"),
    ("ababab", ["aba", "bab"], "<b>ababab</b>", "chồng lấn lệch một"),
    ("xyz", ["xy", "yz"], "<b>xyz</b>", "chồng lấn một ký tự"),
    ("abcde", ["ab", "de"], "<b>ab</b>c<b>de</b>", "khoảng trống đúng một ký tự"),
]
for s, words, expected, note in edge_cases:
    assert oracle(s, words) == expected, ("oracle", note, oracle(s, words))
    for name, fn in solvers.items():
        assert fn(s, words) == expected, (name, note, fn(s, words))
print(f"{len(edge_cases)} edge case: OK")

20 edge case: OK


### 12.2. Phản ví dụ cho các ý tưởng sai

- **Chỉ gộp khi chồng lấn (`a <= ed`), không gộp khi sát nhau**: sai với `s = "abcd"`, `words = ["ab","cd"]` (đáp án `<b>abcd</b>`).
- **`find` rồi nhảy `len(w)`** (bỏ sót lần xuất hiện chồng lấn): sai với `s = "aaa"`, `words = ["aa"]`: hai đoạn `[0,1]` và `[1,2]` cho `<b>aaa</b>`, nhưng nhảy 2 chỉ tìm được `[0,1]` → `<b>aa</b>a`.
- **`replace` từng từ bằng `<b>từ</b>`**: sinh thẻ lồng nhau / làm hỏng chuỗi đã chèn thẻ.

In [29]:
def wrong_no_adjacent(s, words):
    wset = set(words)
    n = len(s)
    pairs = [[i, j] for i in range(n) for j in range(i, n) if s[i:j + 1] in wset]
    merged = []
    for a, b in pairs:
        if merged and a <= merged[-1][1]:          # thiếu "+ 1": không gộp khi sát nhau
            merged[-1][1] = max(merged[-1][1], b)
        else:
            merged.append([a, b])
    out, prev = [], 0
    for a, b in merged:
        out.append(s[prev:a] + "<b>" + s[a:b + 1] + "</b>")
        prev = b + 1
    out.append(s[prev:])
    return "".join(out)

def wrong_skip(s, words):
    bold = [False] * len(s)
    for w in words:
        p = s.find(w)
        while p != -1:
            bold[p:p + len(w)] = [True] * len(w)
            p = s.find(w, p + len(w))              # nhảy len(w): bỏ sót lần chồng lấn
    return render(s, bold)

def wrong_replace(s, words):
    for w in words:
        s = s.replace(w, "<b>" + w + "</b>")
    return s

cases = [("abcd", ["ab", "cd"]), ("aaa", ["aa"]), ("aaabbb", ["aa", "b"])]
for s, w in cases:
    print(f"s={s!r} w={w} | đúng={oracle(s, w)!r}\n    không gộp sát nhau={wrong_no_adjacent(s, w)!r}\n    find nhảy len(w)={wrong_skip(s, w)!r}\n    replace={wrong_replace(s, w)!r}")
assert wrong_no_adjacent(*cases[0]) != oracle(*cases[0])
assert wrong_skip(*cases[1]) != oracle(*cases[1])
assert wrong_replace(*cases[2]) != oracle(*cases[2])

s='abcd' w=['ab', 'cd'] | đúng='<b>abcd</b>'
    không gộp sát nhau='<b>ab</b><b>cd</b>'
    find nhảy len(w)='<b>abcd</b>'
    replace='<b>ab</b><b>cd</b>'
s='aaa' w=['aa'] | đúng='<b>aaa</b>'
    không gộp sát nhau='<b>aaa</b>'
    find nhảy len(w)='<b>aa</b>a'
    replace='<b>aa</b>a'
s='aaabbb' w=['aa', 'b'] | đúng='<b>aaabbb</b>'
    không gộp sát nhau='<b>aaa</b><b>b</b><b>b</b><b>b</b>'
    find nhảy len(w)='<b>aa</b>a<b>bbb</b>'
    replace='<<b>b</b>>aa</<b>b</b>>a<b>b</b><b>b</b><b>b</b>'


### 12.3. Đối chiếu mọi cách bằng vét cạn toàn bộ input nhỏ

Duyệt **mọi** chuỗi `s` trên `{a, b}` có độ dài `1..7` và với mỗi `s`: từ điển rỗng, từ điển gồm mọi từ độ dài `1..3`, cùng 25 tập con ngẫu nhiên (có seed) của chúng.

In [30]:
from itertools import product

all_words = ["".join(p) for n in range(1, 4) for p in product("ab", repeat=n)]
rng = random.Random(616)
checked = 0
for n in range(1, 8):
    for tup in product("ab", repeat=n):
        s = "".join(tup)
        dicts = [[], all_words]
        for _ in range(25):
            dicts.append(rng.sample(all_words, rng.randint(1, 5)))
        for words in dicts:
            expected = oracle(s, words)
            for name, fn in solvers.items():
                assert fn(s, words) == expected, (name, s, words)
            checked += 1
print("đã kiểm", checked, "cặp (s, words): tất cả khớp oracle")

đã kiểm 6858 cặp (s, words): tất cả khớp oracle


### 12.4. Test ngẫu nhiên có seed (đúng ràng buộc)

`s` độ dài `1..14` trên bảng chữ cái nhỏ (có cả chữ hoa và chữ số), `words` gồm `0..6` từ **khác nhau** có độ dài `1..4`. Hàm `is_valid_output` kiểm tra **tính chất** của đầu ra: bỏ thẻ ra thì được `s`; thẻ cân đối, không lồng; không có `</b><b>` (hai khối sát nhau).

In [31]:
def is_valid_output(s, out):
    if out.replace("<b>", "").replace("</b>", "") != s:
        return False
    depth = 0
    i = 0
    while i < len(out):
        if out.startswith("<b>", i):
            if depth:
                return False
            depth = 1
            i += 3
        elif out.startswith("</b>", i):
            if not depth:
                return False
            depth = 0
            i += 4
        else:
            i += 1
    return depth == 0 and "</b><b>" not in out

rng = random.Random(6161)
LET = "aAb1"

def gen(rng):
    alpha = rng.choice(["ab", "abc", "aAb", "ab1", LET])
    s = "".join(rng.choice(alpha) for _ in range(rng.randint(1, 14)))
    words = set()
    for _ in range(rng.randint(0, 6)):
        words.add("".join(rng.choice(alpha) for _ in range(rng.randint(1, 4))))
    words = list(words)
    rng.shuffle(words)
    assert 1 <= len(s) <= 1000 and len(words) <= 100 and len(set(words)) == len(words)
    return s, words

with_tags = 0
for _ in range(4000):
    s, words = gen(rng)
    expected = oracle(s, words)
    assert is_valid_output(s, expected)
    with_tags += "<b>" in expected
    for name, fn in solvers.items():
        assert fn(s, words) == expected, (name, s, words)
print("4000 test ngẫu nhiên: OK;", with_tags, "ca có thẻ,", 4000 - with_tags, "ca không có thẻ")

4000 test ngẫu nhiên: OK; 2530 ca có thẻ, 1470 ca không có thẻ


### 12.5. Test lớn và kiểm tra tính chất

Với `n = 1000` ta không dùng oracle `O(n³)`; ta so các cách nhanh với nhau và kiểm tra `is_valid_output`, cộng thêm vài trường hợp tệ nhất cho từng cách.

In [32]:
rng = random.Random(61600)
fast = ["mask", "trie", "reach", "aho", "leetcode"]
for trial in range(20):
    n = rng.choice([100, 500, 1000])
    alpha = rng.choice(["ab", "abc", "abcdefghijklmnopqrstuvwxyz0123456789"])
    s = "".join(rng.choice(alpha) for _ in range(n))
    words = set()
    for _ in range(rng.choice([0, 5, 50, 100])):
        L = rng.choice([1, 2, 3, 5, 10, 50])
        if rng.random() < 0.5:
            i = rng.randrange(n)
            words.add(s[i:i + L])                      # chắc chắn là chuỗi con của s
        else:
            words.add("".join(rng.choice(alpha) for _ in range(L)))
    words = [w for w in words if w][:100]
    outs = {solvers[name](s, words) for name in fast}
    assert len(outs) == 1, trial
    assert is_valid_output(s, outs.pop()), trial
print("20 test lớn: 5 solver nhất quán và đầu ra hợp lệ")

# tệ nhất: s toàn 'a', từ 'a' * k (nhiều lần xuất hiện chồng lấn), từ rất dài không khớp
s = "a" * 1000
words = ["a" * k for k in range(1, 100)] + ["a" * 999 + "b"]
res = {solvers[name](s, words) for name in fast}
assert res == {"<b>" + s + "</b>"}
# từ dài bằng s nhưng không khớp
words = ["b" * 1000, "a" * 999 + "b"]
assert all(solvers[name](s, words) == s for name in fast)
print("trường hợp tệ nhất: OK")

20 test lớn: 5 solver nhất quán và đầu ra hợp lệ


trường hợp tệ nhất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ gộp khi chồng lấn, quên gộp khi sát nhau

**Code sai:**

In [33]:
class WrongSolution1:
    def addBoldTag(self, s, words):
        return wrong_no_adjacent(s, words)

print("sai  :", WrongSolution1().addBoldTag("abcd", ["ab", "cd"]))
print("đúng :", Solution().addBoldTag("abcd", ["ab", "cd"]))
print("sai  :", WrongSolution1().addBoldTag("aaabbb", ["aa", "b"]))
print("đúng :", Solution().addBoldTag("aaabbb", ["aa", "b"]))

sai  : <b>ab</b><b>cd</b>
đúng : <b>abcd</b>
sai  : <b>aaa</b><b>b</b><b>b</b><b>b</b>
đúng : <b>aaabbb</b>


**Vì sao sai:** đề yêu cầu "nếu hai đoạn bọc **liền kề**, gộp lại". Điều kiện gộp là `a <= ed + 1`, không phải `a <= ed`. **Code đúng:** `if merged and a <= merged[-1][1] + 1`. (Hoặc dùng mask: các ô `True` liền nhau tự gộp.)

### Lỗi 2: Bỏ sót các lần xuất hiện chồng lấn

In [34]:
def wrong_find_skip(s, words):
    bold = [False] * len(s)
    for w in words:
        p = s.find(w)
        while p != -1:
            bold[p:p + len(w)] = [True] * len(w)
            p = s.find(w, p + len(w))           # sai: nhảy qua cả từ
    return render(s, bold)

for s, words in [("aaa", ["aa"]), ("ababa", ["aba"]), ("aaaa", ["aaa"])]:
    print(f"{s!r} {words} | sai: {wrong_find_skip(s, words)!r} | đúng: {Solution().addBoldTag(s, words)!r}")

'aaa' ['aa'] | sai: '<b>aa</b>a' | đúng: '<b>aaa</b>'
'ababa' ['aba'] | sai: '<b>aba</b>ba' | đúng: '<b>ababa</b>'
'aaaa' ['aaa'] | sai: '<b>aaa</b>a' | đúng: '<b>aaaa</b>'


**Vì sao sai:** `"aaa"` chứa `"aa"` ở vị trí 0 **và** 1. Nếu sau khi tìm thấy ở `p` ta nhảy tới `p + len(w)`, ta bỏ lỡ vị trí `p + 1`. **Code đúng:** tìm tiếp từ `p + 1`.

### Lỗi 3: Dùng `str.replace` để bọc từng từ

In [35]:
def wrong_replace_solution(s, words):
    for w in words:
        s = s.replace(w, "<b>" + w + "</b>")
    return s

for s, words in [("aaabbb", ["aa", "b"]), ("abcxyz123", ["abc", "123"]), ("abab", ["ab", "b"])]:
    print(f"{s!r} {words} | sai: {wrong_replace_solution(s, words)!r} | đúng: {Solution().addBoldTag(s, words)!r}")

'aaabbb' ['aa', 'b'] | sai: '<<b>b</b>>aa</<b>b</b>>a<b>b</b><b>b</b><b>b</b>' | đúng: '<b>aaabbb</b>'
'abcxyz123' ['abc', '123'] | sai: '<b>abc</b>xyz<b>123</b>' | đúng: '<b>abc</b>xyz<b>123</b>'
'abab' ['ab', 'b'] | sai: '<<b>b</b>>a<b>b</b></<b>b</b>><<b>b</b>>a<b>b</b></<b>b</b>>' | đúng: '<b>abab</b>'


**Vì sao sai:** (1) `replace` không biết gộp các đoạn chồng/sát nhau: ra `<b>aa</b>a<b>b</b>...`; (2) khi từ thứ hai là `b`, nó cũng khớp **chữ `b` trong thẻ `<b>`** vừa chèn, làm hỏng thẻ. **Code đúng:** không sửa `s` trực tiếp; tìm vị trí trên `s` gốc (mask/khoảng) rồi dựng chuỗi mới ở bước cuối.

### Lỗi 4: Quên trường hợp `words` rỗng (hoặc không có khoảng nào)

In [36]:
class WrongSolution4:
    def addBoldTag(self, s, words):
        root = {}
        for w in words:
            node = root
            for c in w:
                node = node.setdefault(c, {})
            node["$"] = True
        pairs = []
        for i in range(len(s)):
            node = root
            for j in range(i, len(s)):
                if s[j] not in node:
                    break
                node = node[s[j]]
                if "$" in node:
                    pairs.append([i, j])
        st, ed = pairs[0]                       # IndexError khi pairs rỗng
        return "..."

try:
    WrongSolution4().addBoldTag("abc", ["x"])
except IndexError as e:
    print("lỗi:", type(e).__name__, "-", e)
print("đúng:", Solution().addBoldTag("abc", ["x"]), "|", Solution().addBoldTag("abc", []))

lỗi: IndexError - list index out of range
đúng: abc | abc


**Vì sao sai:** lấy `pairs[0]` khi danh sách rỗng gây `IndexError`. **Code đúng:** `if not pairs: return s` ngay sau khi quét (hoặc dùng mask để khỏi phải xử lý riêng).

### Lỗi 5: Chèn thẻ cho từng lần xuất hiện (thẻ lồng nhau)

In [37]:
def wrong_nested(s, words):
    n = len(s)
    opens, closes = [0] * (n + 1), [0] * (n + 1)
    for w in words:
        for i in range(n):
            if s.startswith(w, i):
                opens[i] += 1
                closes[i + len(w)] += 1
    out = []
    for k in range(n + 1):
        out.append("</b>" * closes[k])
        out.append("<b>" * opens[k])
        if k < n:
            out.append(s[k])
    return "".join(out)

print("sai  :", wrong_nested("aaabbb", ["aa", "b"]))
print("đúng :", Solution().addBoldTag("aaabbb", ["aa", "b"]))

sai  : <b>a<b>a</b>a</b><b>b</b><b>b</b><b>b</b>
đúng : <b>aaabbb</b>


**Vì sao sai:** đề yêu cầu **gộp** thành các khối thẻ phẳng chứ không phải mỗi lần xuất hiện một cặp thẻ (đó chính là kết quả "chưa gộp" ở phần giải thích của đề). **Code đúng:** hợp nhất theo mask hoặc merge intervals.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy **"bọc/tô đậm các đoạn, gộp khi chồng hoặc sát nhau"** → đổi sang **mảng đánh dấu từng ký tự** rồi đọc theo khối liên tiếp; tránh phải nghĩ về từng khoảng.
- Bài **merge intervals** chuẩn dùng `a <= ed`; bài này **`a <= ed + 1`** vì "sát nhau" cũng gộp (khoảng đóng trên chỉ số nguyên). Hãy luôn tự hỏi: hai khoảng đóng `[0,2]` và `[3,5]` có "chạm" nhau không?
- **Không sửa `s` tại chỗ** khi đang tìm kiếm (thẻ chèn vào sẽ làm lệch chỉ số và gây khớp giả).
- Tìm **mọi lần xuất hiện, kể cả chồng lấn**: `find` tiếp từ `p + 1`, hoặc `startswith` tại mọi `i`.
- Nhiều từ + một chuỗi → nghĩ **trie** (nếu `n` nhỏ) hoặc **Aho-Corasick** (nếu cần `O(n + T)`).
- Dùng `''.join(list)` để ghép chuỗi; tránh `+=` trong vòng lặp lớn.
- Luôn thử tay: `words` rỗng, từ dài hơn `s`, hai từ sát nhau, từ chồng lấn, hai khối cách nhau đúng một ký tự, phân biệt hoa/thường.

## 15. Biến thể

### 15.1. Thẻ tuỳ ý (ví dụ `<strong>` hoặc `**`) và trả về các khối

Ta tách phần tìm bold thành `bold_mask(s, words)` rồi `render(s, mask, open_tag, close_tag)` — đổi thẻ chỉ là đổi hai tham số. Hàm `bold_blocks` trả về danh sách khối `(start, end)` (đầu và cuối đều tính).

In [38]:
def bold_mask(s, words):
    n = len(s)
    reach = [0] * n
    for i in range(n):
        for w in words:
            if s.startswith(w, i):
                reach[i] = max(reach[i], i + len(w))
    far, mask = 0, []
    for k in range(n):
        far = max(far, reach[k])
        mask.append(k < far)
    return mask

def bold_blocks(s, words):
    mask, blocks, k, n = bold_mask(s, words), [], 0, len(s)
    while k < n:
        if mask[k]:
            j = k
            while j + 1 < n and mask[j + 1]:
                j += 1
            blocks.append((k, j))
            k = j + 1
        else:
            k += 1
    return blocks

print(render("abcxyz123", bold_mask("abcxyz123", ["abc", "123"]), "<strong>", "</strong>"))
print(render("abcxyz123", bold_mask("abcxyz123", ["abc", "123"]), "**", "**"))
print(bold_blocks("abcxyz123", ["abc", "123"]), bold_blocks("aaabbb", ["aa", "b"]))

rng = random.Random(151)
for _ in range(1500):
    s, w = gen(rng)
    assert render(s, bold_mask(s, w)) == oracle(s, w)
print("bold_mask + render khớp oracle")

<strong>abc</strong>xyz<strong>123</strong>
**abc**xyz**123**
[(0, 2), (6, 8)] [(0, 5)]
bold_mask + render khớp oracle


### 15.2. Đếm số ký tự bị tô đậm và số khối

Bài này hay xuất hiện dưới dạng "bao nhiêu ký tự bị tô?" — chỉ cần `sum(mask)`; "bao nhiêu khối?" = số lần `mask` chuyển từ `False` sang `True` — đúng bằng số thẻ `<b>` trong đáp án.

In [39]:
def bold_stats(s, words):
    mask = bold_mask(s, words)
    chars = sum(mask)
    blocks = sum(1 for k in range(len(s)) if mask[k] and (k == 0 or not mask[k - 1]))
    return chars, blocks

print(bold_stats("aaabbb", ["aa", "b"]), bold_stats("abcxyz123", ["abc", "123"]), bold_stats("abc", []))
rng = random.Random(152)
for _ in range(1500):
    s, w = gen(rng)
    chars, blocks = bold_stats(s, w)
    out = oracle(s, w)
    assert blocks == out.count("<b>") and len(out) == len(s) + 7 * blocks
    inside = out.replace("</b>", "<b>").split("<b>")[1::2]
    assert chars == sum(len(x) for x in inside)
print("bold_stats khớp số thẻ và độ dài đầu ra")

(6, 1) (6, 2) (0, 0)
bold_stats khớp số thẻ và độ dài đầu ra


### 15.3. Bài "bỏ thẻ" (nghịch đảo): khôi phục `s` và các từ tối thiểu

Nghịch đảo đơn giản: bỏ mọi thẻ khỏi đầu ra phải ra đúng `s` (đã dùng làm tính chất kiểm thử). Việc tìm lại `words` thì không duy nhất — có nhiều `words` cho cùng đầu ra — nên đó là bài mở.

### 15.4. Các bài liên quan trên LeetCode (đọc thêm)

- **758. Bold Words in String:** y hệt bài này (bản công khai).
- **56. Merge Intervals / 57. Insert Interval:** phần gộp khoảng (nhưng gộp khi `a <= ed`).
- **1065. Index Pairs of a String:** liệt kê mọi cặp `[i, j]` mà `s[i..j]` là một từ — chính là bước "tìm khoảng" của bài này.
- **208. Implement Trie, 212. Word Search II:** nền tảng trie.
- **616/758 và Aho-Corasick:** bài toán tìm nhiều mẫu — xem thêm các bài như 1032. Stream of Characters.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi chuỗi con + gộp khoảng | tra `set` cho từng `s[i:j+1]` | `O(n² · L)` | `O(n²)` | chỉ để hiểu đề / oracle |
| 2. Mask theo từng từ | `find` lặp, gán lát cắt `True` | `O(W · n · L)` | `O(n)` | **dễ viết nhất**, đủ nhanh với đề này |
| 3. Trie + gộp khoảng | mọi từ bắt đầu ở `i` trong một lượt | `O(T + n · L)` | `O(T + n)` | **cách nộp bài chuẩn** (giống lời giải mẫu) |
| 4. `reach` + running max | chỉ cần lần khớp dài nhất tại mỗi `i` | `O(n · W · L)` | `O(n)` | muốn chèn thẻ ngay lúc quét, không mask/merge |
| 5. Aho-Corasick | quét `s` một lần, `fail` link | `O(T + n)` | `O(T + n)` | nhiều từ, `s` rất dài, bài tìm nhiều mẫu |

## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán rồi chạy:

In [40]:
print(Solution().addBoldTag("abcde", ["bc", "cd"]))
print(Solution().addBoldTag("abcde", ["ab", "de"]))
print(Solution().addBoldTag("aaa", ["a"]))
print(Solution().addBoldTag("abc", ["d"]))

a<b>bcd</b>e
<b>ab</b>c<b>de</b>
<b>aaa</b>
abc


**Bài 2 (tính tay).** Với `s = "xaabbx"`, `words = ["aa", "ab", "bx"]`: liệt kê mọi khoảng `[i, j]` khớp một từ, rồi cho biết các khối bold sau khi gộp. Gán danh sách các khối `(đầu, cuối)` vào `ans2` (đang là `None`), ví dụ `[(1, 3)]`.

In [41]:
ans2 = None      # TODO: list các tuple (đầu, cuối) của khối bold, theo thứ tự tăng dần

**Bài 3 (điền TODO).** Hoàn thành `merge_touching(pairs)`: cho list khoảng `[a, b]` (đóng hai đầu, **đã sắp theo `a`**), gộp những khoảng chồng lấn hoặc sát nhau; trả về list khoảng đã gộp. Ví dụ `[[0,1],[2,3],[5,5]]` -> `[[0,3],[5,5]]`.

In [42]:
def merge_touching(pairs):
    merged = []
    for a, b in pairs:
        # TODO: nếu merged không rỗng và a <= merged[-1][1] + 1 thì kéo dài khoảng cuối, ngược lại thêm [a, b]
        pass
    return merged

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_add_bold(s, words)` theo **Cách 2** (mảng mask + `find` lặp từ `p + 1`), trả về chuỗi có thẻ `<b>`.

In [43]:
def my_add_bold(s, words):
    # TODO
    pass

**Bài 5 (mini challenge).** (a) `add_tag(s, words, tag)`: giống bài gốc nhưng thẻ tuỳ ý, ví dụ `tag = "i"` bọc bằng `<i>...</i>`. (b) `longest_bold_run(s, words)`: độ dài của **khối bold dài nhất** (0 nếu không có). Ví dụ `longest_bold_run("abcxyz123", ["abc","xyz"])` = 6 (vì `abc` và `xyz` sát nhau).

In [44]:
def add_tag(s, words, tag):
    # TODO
    pass

def longest_bold_run(s, words):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: ghi ra các ký tự nào được tô; các khối tô liền nhau thì gộp.
- Bài 2: tìm các từ `aa`, `ab`, `bx` trong `xaabbx`, ghi vị trí đầu cuối.
- Bài 3: giống `merge_intervals` ở mục 3.5.
- Bài 4: nhớ `bold = [False] * n` và `s.find(w, p + 1)`.
- Bài 5: `add_tag` chỉ đổi chuỗi thẻ; `longest_bold_run` duyệt mask tìm dãy `True` dài nhất.

**Mức 2 (vừa):**
- Bài 2: `aa` tại `[1,2]`, `ab` tại `[2,3]`, `bx` tại `[4,5]`; `[1,2]` với `[2,3]` chồng lấn → `[1,3]`; `[1,3]` và `[4,5]` sát nhau (`4 = 3 + 1`) → `[1,5]`.
- Bài 3: `if merged and a <= merged[-1][1] + 1: merged[-1][1] = max(merged[-1][1], b)` rồi `else: merged.append([a, b])`.
- Bài 4: sau khi có mask, dùng vòng `while` mở rộng khối `True` để chèn thẻ (mục 3.4).
- Bài 5: dùng `bold_mask` ở mục 15.1.

**Mức 3 (gần đáp án):**
- Bài 3: xem mục 3.5.
- Bài 4: xem Cách 2 ở mục 7.1.
- Bài 5: `longest_bold_run`: `best = cur = 0; for m in mask: cur = cur + 1 if m else 0; best = max(best, cur)`.

## 19. Đáp án tham khảo

In [45]:
# Bài 1: "a<b>bcd</b>e", "<b>ab</b>c<b>de</b>", "<b>aaa</b>", "abc"
assert Solution().addBoldTag("abcde", ["bc", "cd"]) == "a<b>bcd</b>e"
assert Solution().addBoldTag("abcde", ["ab", "de"]) == "<b>ab</b>c<b>de</b>"
assert Solution().addBoldTag("aaa", ["a"]) == "<b>aaa</b>"
assert Solution().addBoldTag("abc", ["d"]) == "abc"
print("Bài 1 OK")

# Bài 2
ans2_ref = [(1, 5)]
assert bold_blocks("xaabbx", ["aa", "ab", "bx"]) == ans2_ref
print("Bài 2 OK:", ans2_ref, "->", oracle("xaabbx", ["aa", "ab", "bx"]))

Bài 1 OK
Bài 2 OK: [(1, 5)] -> x<b>aabbx</b>


In [46]:
# Bài 3
def merge_touching_ref(pairs):
    merged = []
    for a, b in pairs:
        if merged and a <= merged[-1][1] + 1:
            merged[-1][1] = max(merged[-1][1], b)
        else:
            merged.append([a, b])
    return merged

assert merge_touching_ref([[0, 1], [2, 3], [5, 5]]) == [[0, 3], [5, 5]]
assert merge_touching_ref([]) == []
rng = random.Random(1903)
for _ in range(2000):
    pairs = sorted([a, a + rng.randint(0, 4)] for a in [rng.randint(0, 15) for _ in range(rng.randint(0, 8))])
    got = merge_touching_ref([p[:] for p in pairs])
    covered = set(k for a, b in pairs for k in range(a, b + 1))
    from_got = set(k for a, b in got for k in range(a, b + 1))
    assert covered == from_got
    assert all(got[i][1] + 1 < got[i + 1][0] for i in range(len(got) - 1))     # các khối thật sự tách rời
print("Bài 3 OK")

# Bài 4
def my_add_bold_ref(s, words):
    bold = [False] * len(s)
    for w in words:
        p = s.find(w)
        while p != -1:
            bold[p:p + len(w)] = [True] * len(w)
            p = s.find(w, p + 1)
    return render(s, bold)

rng = random.Random(1904)
for _ in range(2000):
    s, w = gen(rng)
    assert my_add_bold_ref(s, w) == oracle(s, w)
print("Bài 4 OK")

Bài 3 OK


Bài 4 OK


In [47]:
# Bài 5
def add_tag_ref(s, words, tag):
    return render(s, bold_mask(s, words), f"<{tag}>", f"</{tag}>")

def longest_bold_run_ref(s, words):
    best = cur = 0
    for m in bold_mask(s, words):
        cur = cur + 1 if m else 0
        best = max(best, cur)
    return best

assert add_tag_ref("abcxyz123", ["abc", "123"], "i") == "<i>abc</i>xyz<i>123</i>"
assert longest_bold_run_ref("abcxyz123", ["abc", "xyz"]) == 6
assert longest_bold_run_ref("abc", []) == 0
rng = random.Random(1905)
for _ in range(2000):
    s, w = gen(rng)
    out = oracle(s, w)
    assert add_tag_ref(s, w, "b") == out
    runs = [len(x) for x in out.replace("</b>", "<b>").split("<b>")[1::2]]
    assert longest_bold_run_ref(s, w) == (max(runs) if runs else 0)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template mask (dễ nhớ nhất):**

```python
class Solution:
    def addBoldTag(self, s, words):
        n = len(s)
        bold = [False] * n
        for w in words:
            p = s.find(w)
            while p != -1:
                bold[p:p + len(w)] = [True] * len(w)
                p = s.find(w, p + 1)          # +1: cho phép chồng lấn
        out, i = [], 0
        while i < n:
            if bold[i]:
                j = i
                while j < n and bold[j]:
                    j += 1
                out.append("<b>" + s[i:j] + "</b>")
                i = j
            else:
                out.append(s[i])
                i += 1
        return "".join(out)
```

**Template gộp khoảng (sát nhau cũng gộp):**

```python
merged = []
for a, b in pairs:                       # pairs sắp theo a
    if merged and a <= merged[-1][1] + 1:
        merged[-1][1] = max(merged[-1][1], b)
    else:
        merged.append([a, b])
```

**Câu thần chú:** *tô từng ký tự rồi đọc theo khối; chồng lấn và sát nhau tự gộp; tìm tiếp từ `p + 1`; không sửa `s` khi đang tìm.*

**Checklist trước khi submit:**

- [ ] Gộp cả khi **sát nhau** (`a <= ed + 1`) chứ không chỉ khi chồng lấn?
- [ ] Tìm mọi lần xuất hiện, kể cả chồng lấn (`find` tiếp từ `p + 1`)?
- [ ] `words` rỗng / không khớp từ nào → trả về đúng `s`?
- [ ] Không dùng `replace` trên chuỗi đã chèn thẻ; không có thẻ lồng nhau?
- [ ] Đã thử: hai từ sát nhau, từ bao từ, từ dài hơn `s`, hai khối cách nhau 1 ký tự, hoa/thường, chữ số?
- [ ] Ghép chuỗi bằng `''.join(...)`?
- [ ] Bài Premium: đã đối chiếu lại đề gốc (nếu có tài khoản)?